In [2]:
import nbloss

print("Imported from:", nbloss.__file__)

Imported from: C:\Users\koeng\Downloads\nbloss-main\nbloss-main\nbloss\__init__.py


# Step 1 — Load and inspect the Framingham dataset

# Framingham Data Preparation

This script constructs the dataset used for the 15-year Framingham ANYCHD prediction experiment.

Starting from the original longitudinal Framingham dataset, it:

- Retains the baseline examination (`PERIOD == 1`).
- Excludes participants with prevalent coronary heart disease at baseline.
- Defines incident ANYCHD within 15 years as the binary outcome.
- Excludes participants whose 15-year outcome cannot be determined because of insufficient follow-up.
- Selects the baseline predictors used in the modelling experiments.
- Creates the final modelling dataset without participant identifiers.

The resulting dataset is used to compare conventional NLL/BCE training with Smooth Net Benefit training. 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Step 1 — Load and prepare Framingham data
# ============================================================

DATA_PATH = Path("Data") / "framingham.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    sep=None,
    engine="python",
)

df_raw = df_raw.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)


# ------------------------------------------------------------
# Keep baseline examination only
# ------------------------------------------------------------

df = df_raw.loc[
    df_raw["PERIOD"] == 1
].copy()

assert df["RANDID"].is_unique


# ------------------------------------------------------------
# Exclude prevalent coronary heart disease
# ------------------------------------------------------------

df = df.loc[
    df["PREVCHD"] == 0
].copy()

assert len(df) == 4240


# ------------------------------------------------------------
# Construct 15-year incident ANYCHD outcome
# ------------------------------------------------------------

HORIZON_YEARS = 15
HORIZON_DAYS = 365.25 * HORIZON_YEARS

event_15y = (
    (df["ANYCHD"] == 1)
    & (df["TIMECHD"] <= HORIZON_DAYS)
)

non_event_15y = (
    ~event_15y
    & (df["TIMECHD"] >= HORIZON_DAYS)
)

known_outcome = (
    event_15y
    | non_event_15y
)

df = df.loc[
    known_outcome
].copy()

df["FifteenYearCHD"] = (
    event_15y.loc[df.index]
    .astype(int)
)


# ------------------------------------------------------------
# Retain modelling variables
# ------------------------------------------------------------

PREDICTOR_COLS = [
    "SEX",
    "TOTCHOL",
    "AGE",
    "SYSBP",
    "DIABP",
    "CURSMOKE",
    "CIGPDAY",
    "BMI",
    "DIABETES",
    "BPMEDS",
    "HEARTRTE",
    "GLUCOSE",
    "educ",
    "PREVSTRK",
    "PREVHYP",
]

TARGET_COL = "FifteenYearCHD"

model_df = df[
    PREDICTOR_COLS + [TARGET_COL]
].copy()


# ------------------------------------------------------------
# Ensure modelling variables are numeric
# ------------------------------------------------------------

for col in model_df.columns:
    model_df[col] = pd.to_numeric(
        model_df[col]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert model_df[TARGET_COL].isin([0, 1]).all()

print("Modelling dataset shape:", model_df.shape)
print(
    "15-year ANYCHD prevalence:",
    f"{model_df[TARGET_COL].mean():.3%}",
)

print("\nMissing values:")
print(
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

# Step 2 — Create repeated train/test splits and preprocess the data

This step creates the five rotating 80/20 train/test splits used for the Framingham logistic-regression experiment. All preprocessing steps are fitted on the training data only and then applied to the corresponding test fold. No validation set is used. Two preprocessing variants are available: `mild`, where education is retained as a single ordinal variable, and `moderate`, where education is one-hot encoded.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


# ============================================================
# Step 2 — Framingham XGBoost preprocessing
# New 15-year incident ANYCHD analysis
# ============================================================


@dataclass
class SplitPack:
    run_id: int
    seed: int
    feature_cols: list

    X_train: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray

    X_train_t: torch.Tensor
    X_test_t: torch.Tensor
    y_train_t: torch.Tensor
    y_test_t: torch.Tensor

    imputer_bin: SimpleImputer
    imputer_cont: SimpleImputer
    scaler_cont: StandardScaler

    binary_cols: list
    categorical_cols: list
    continuous_cols: list
    engineered_cont_cols: list
    ohe_cols: list

    y_train_mean: float
    y_test_mean: float

    fe_level: str = "mild"


# ============================================================
# Variable definitions
# ============================================================

BINARY_COLS = [
    "SEX",
    "CURSMOKE",
    "DIABETES",
    "BPMEDS",
    "PREVSTRK",
    "PREVHYP",
]

CATEGORICAL_COLS = [
    "educ",
]

CONTINUOUS_COLS = [
    "AGE",
    "CIGPDAY",
    "TOTCHOL",
    "SYSBP",
    "DIABP",
    "BMI",
    "HEARTRTE",
    "GLUCOSE",
]


# ============================================================
# Validation
# ============================================================

def _validate_columns(
    df: pd.DataFrame,
    target_col: str,
):
    required = set(
        BINARY_COLS
        + CATEGORICAL_COLS
        + CONTINUOUS_COLS
        + [target_col]
    )

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )


# ============================================================
# Prepare base dataframe
# ============================================================

def _prepare_base_frame(
    df: pd.DataFrame,
    target_col: str,
) -> pd.DataFrame:

    cols = (
        BINARY_COLS
        + CATEGORICAL_COLS
        + CONTINUOUS_COLS
        + [target_col]
    )

    out = df[cols].copy()

    return out


# ============================================================
# Optional feature engineering
#
# Currently deliberately empty.
# This preserves the structure of the previous pipeline
# without adding arbitrary new predictors.
# ============================================================

def _add_engineered_features_after_imputation(
    X_cont_df: pd.DataFrame,
    X_bin_df: pd.DataFrame,
    fe_level: str,
):

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'"
        )

    X_eng = pd.DataFrame(
        index=X_cont_df.index
    )

    engineered_cols = []

    return X_eng, engineered_cols


# ============================================================
# Education:
# no OHE
# ============================================================

def _education_no_ohe_train_test(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    imputer_edu = SimpleImputer(
        strategy="most_frequent"
    )

    X_train_edu = pd.DataFrame(
        imputer_edu.fit_transform(
            X_train_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_train_cat.index,
    ).astype(float)

    X_test_edu = pd.DataFrame(
        imputer_edu.transform(
            X_test_cat[["educ"]]
        ),
        columns=["educ"],
        index=X_test_cat.index,
    ).astype(float)

    return (
        X_train_edu,
        X_test_edu,
        ["educ"],
        imputer_edu,
    )


# ============================================================
# Education:
# one-hot encoding
# ============================================================

def _ohe_education_train_test(
    X_train_cat: pd.DataFrame,
    X_test_cat: pd.DataFrame,
):

    X_train_cat = X_train_cat.copy()
    X_test_cat = X_test_cat.copy()

    for frame in (
        X_train_cat,
        X_test_cat,
    ):
        frame["educ"] = (
            frame["educ"]
            .astype("object")
            .where(
                ~frame["educ"].isna(),
                "missing",
            )
        )

        frame["educ"] = (
            frame["educ"]
            .astype(str)
        )

    train_ohe = pd.get_dummies(
        X_train_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    test_ohe = pd.get_dummies(
        X_test_cat,
        columns=["educ"],
        prefix="educ",
        drop_first=False,
    )

    # Only categories observed in training define the feature space
    ohe_cols = train_ohe.columns.tolist()

    test_ohe = test_ohe.reindex(
        columns=ohe_cols,
        fill_value=0.0,
    )

    train_ohe = train_ohe.astype(float)
    test_ohe = test_ohe.astype(float)

    return (
        train_ohe,
        test_ohe,
        ohe_cols,
    )


# ============================================================
# Create 5 rotating stratified folds
# ============================================================

def make_framingham_splits_5x(
    df: pd.DataFrame,
    *,
    target_col: str = "FifteenYearCHD",
    base_seed: int = 42,
    n_runs: int = 5,
    device: str = "cpu",
    fe_level: str = "mild",
):

    if fe_level not in (
        "mild",
        "moderate",
    ):
        raise ValueError(
            "fe_level must be 'mild' or 'moderate'"
        )

    if n_runs != 5:
        raise ValueError(
            "This regime uses exactly 5 rotating folds, "
            "so n_runs must be 5."
        )

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    _validate_columns(
        df,
        target_col,
    )

    df_base = _prepare_base_frame(
        df,
        target_col,
    )

    # --------------------------------------------------------
    # Outcome
    # --------------------------------------------------------

    y_all = (
        df_base[target_col]
        .to_numpy(dtype=int)
    )

    if not np.isin(
        y_all,
        [0, 1],
    ).all():
        raise ValueError(
            "Target must contain only 0 and 1."
        )

    X_df = (
        df_base
        .drop(columns=[target_col])
        .copy()
    )

    # --------------------------------------------------------
    # Stratified 5-fold CV
    # --------------------------------------------------------

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=base_seed,
    )

    splits = []

    for k, (
        train_idx,
        test_idx,
    ) in enumerate(
        skf.split(
            X_df,
            y_all,
        )
    ):

        seed = base_seed + k

        # ----------------------------------------------------
        # Raw split
        # ----------------------------------------------------

        X_train_df = (
            X_df
            .iloc[train_idx]
            .copy()
        )

        X_test_df = (
            X_df
            .iloc[test_idx]
            .copy()
        )

        y_train = y_all[train_idx]
        y_test = y_all[test_idx]

        # ----------------------------------------------------
        # Separate variable types
        # ----------------------------------------------------

        X_train_bin_raw = (
            X_train_df[BINARY_COLS]
            .copy()
        )

        X_test_bin_raw = (
            X_test_df[BINARY_COLS]
            .copy()
        )

        X_train_cat = (
            X_train_df[CATEGORICAL_COLS]
            .copy()
        )

        X_test_cat = (
            X_test_df[CATEGORICAL_COLS]
            .copy()
        )

        X_train_cont = (
            X_train_df[CONTINUOUS_COLS]
            .copy()
        )

        X_test_cont = (
            X_test_df[CONTINUOUS_COLS]
            .copy()
        )

        # ====================================================
        # Binary variables
        # ====================================================

        imputer_bin = SimpleImputer(
            strategy="most_frequent"
        )

        X_train_bin = pd.DataFrame(
            imputer_bin.fit_transform(
                X_train_bin_raw
            ),
            columns=BINARY_COLS,
            index=X_train_bin_raw.index,
        ).astype(float)

        X_test_bin = pd.DataFrame(
            imputer_bin.transform(
                X_test_bin_raw
            ),
            columns=BINARY_COLS,
            index=X_test_bin_raw.index,
        ).astype(float)

        # ====================================================
        # Continuous variables
        # ====================================================

        imputer_cont = SimpleImputer(
            strategy="median"
        )

        scaler_cont = StandardScaler()

        # Fit imputation ONLY on training data
        X_train_cont_i = pd.DataFrame(
            imputer_cont.fit_transform(
                X_train_cont
            ),
            columns=CONTINUOUS_COLS,
            index=X_train_cont.index,
        )

        X_test_cont_i = pd.DataFrame(
            imputer_cont.transform(
                X_test_cont
            ),
            columns=CONTINUOUS_COLS,
            index=X_test_cont.index,
        )

        # ====================================================
        # Optional engineered continuous features
        # ====================================================

        (
            X_train_eng,
            engineered_cont_cols,
        ) = _add_engineered_features_after_imputation(
            X_train_cont_i,
            X_train_bin,
            fe_level,
        )

        (
            X_test_eng,
            _,
        ) = _add_engineered_features_after_imputation(
            X_test_cont_i,
            X_test_bin,
            fe_level,
        )

        X_train_cont_full = pd.concat(
            [
                X_train_cont_i,
                X_train_eng,
            ],
            axis=1,
        )

        X_test_cont_full = pd.concat(
            [
                X_test_cont_i,
                X_test_eng,
            ],
            axis=1,
        )

        cont_full_cols = (
            X_train_cont_full
            .columns
            .tolist()
        )

        # Fit scaling ONLY on training data
        X_train_cont_s = pd.DataFrame(
            scaler_cont.fit_transform(
                X_train_cont_full
            ),
            columns=cont_full_cols,
            index=X_train_cont_full.index,
        )

        X_test_cont_s = pd.DataFrame(
            scaler_cont.transform(
                X_test_cont_full
            ),
            columns=cont_full_cols,
            index=X_test_cont_full.index,
        )

        # ====================================================
        # Education
        # ====================================================

        if fe_level == "mild":

            (
                X_train_cat_enc,
                X_test_cat_enc,
                cat_encoded_cols,
                imputer_edu,
            ) = _education_no_ohe_train_test(
                X_train_cat,
                X_test_cat,
            )

            ohe_cols = []

        else:

            (
                X_train_cat_enc,
                X_test_cat_enc,
                ohe_cols,
            ) = _ohe_education_train_test(
                X_train_cat,
                X_test_cat,
            )

            cat_encoded_cols = ohe_cols
            imputer_edu = None

        # ====================================================
        # Combine final predictors
        # ====================================================

        X_train_final_df = pd.concat(
            [
                X_train_bin,
                X_train_cat_enc,
                X_train_cont_s,
            ],
            axis=1,
        )

        X_test_final_df = pd.concat(
            [
                X_test_bin,
                X_test_cat_enc,
                X_test_cont_s,
            ],
            axis=1,
        )

        feature_cols = (
            X_train_final_df
            .columns
            .tolist()
        )

        # Ensure exact train/test correspondence
        assert (
            X_train_final_df.columns.tolist()
            == X_test_final_df.columns.tolist()
        )

        # Ensure preprocessing removed all missing values
        assert not X_train_final_df.isna().any().any()
        assert not X_test_final_df.isna().any().any()

        # ====================================================
        # NumPy arrays
        # ====================================================

        X_train = (
            X_train_final_df
            .to_numpy(dtype=np.float32)
        )

        X_test = (
            X_test_final_df
            .to_numpy(dtype=np.float32)
        )

        # ====================================================
        # Torch tensors
        # ====================================================

        X_train_t = torch.tensor(
            X_train,
            dtype=torch.float32,
            device=device,
        )

        X_test_t = torch.tensor(
            X_test,
            dtype=torch.float32,
            device=device,
        )

        y_train_t = torch.tensor(
            y_train.reshape(-1, 1),
            dtype=torch.float32,
            device=device,
        )

        y_test_t = torch.tensor(
            y_test.reshape(-1, 1),
            dtype=torch.float32,
            device=device,
        )

        # ====================================================
        # Store split
        # ====================================================

        splits.append(
            SplitPack(
                run_id=k,
                seed=seed,
                feature_cols=feature_cols,

                X_train=X_train,
                X_test=X_test,

                y_train=y_train,
                y_test=y_test,

                X_train_t=X_train_t,
                X_test_t=X_test_t,

                y_train_t=y_train_t,
                y_test_t=y_test_t,

                imputer_bin=imputer_bin,
                imputer_cont=imputer_cont,
                scaler_cont=scaler_cont,

                binary_cols=BINARY_COLS.copy(),
                categorical_cols=CATEGORICAL_COLS.copy(),
                continuous_cols=CONTINUOUS_COLS.copy(),

                engineered_cont_cols=(
                    engineered_cont_cols
                ),

                ohe_cols=ohe_cols,

                y_train_mean=float(
                    np.mean(y_train)
                ),

                y_test_mean=float(
                    np.mean(y_test)
                ),

                fe_level=fe_level,
            )
        )

    return splits

# Step 3 — Configure the logistic-regression experiment

This step defines the global settings for the Framingham logistic-regression experiment. We set the computation device, random seeds, smooth Net Benefit annealing schedule, threshold-band policy, and the simple logistic-regression model used for both BCE training and subsequent SNB fine-tuning.

In [11]:
import random

import numpy as np
import torch
import torch.nn as nn

from nbloss.trainer import set_seed
from nbloss.trainer import nb_anneal_only_with_l2
from nbloss.metrics import average_nb_over_range


DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

SEED_GLOBAL = 1234
MODEL_SEED = 4242

set_seed(SEED_GLOBAL)

LR_ADAMW = 0.001
INVERSE_TEMPS = (1.0, 4.0, 10.0)
EPOCHS_PER_TEMP = 300
PATIENCE_HARD = 20
TRAIN_RANGE_POINTS = 11
TEST_RANGE_POINTS = 201

BAND_HALF_WIDTH = 0.025
MID_PREV_LOW, MID_PREV_HIGH = 0.40, 0.60

L2_LAM_GRID = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]

LOCAL_START_HALF_WIDTH = 0.1
LOCAL_EXPAND_STEP = 0.01
LOCAL_MIN_POS = 50
LOCAL_MIN_NEG = 50

TEMP_MAX_ITER = 500
PLATT_MAX_ITER = 500


def band_from_t_ref(t_ref: float, half_width: float = BAND_HALF_WIDTH) -> tuple[float, float]:
    eps = 1e-9
    t_ref = float(np.clip(t_ref, eps, 1.0 - eps))

    t_min = max(eps, t_ref - float(half_width))
    t_max = min(1.0 - eps, t_ref + float(half_width))

    if not t_min < t_max:
        span = min(t_ref - eps, 1.0 - eps - t_ref, float(half_width))
        t_min = t_ref - span
        t_max = t_ref + span

    return float(t_min), float(t_max)


class TorchLR(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.linear = nn.Linear(d_in, 1, bias=True)

    def forward(self, x):
        return self.linear(x).squeeze(-1)

# Step 4 — Define data loaders and helper losses

This step defines small helper functions used by the logistic-regression training code. The data loader keeps the outcome as a one-dimensional tensor to match the model output shape. The L2 penalty is applied only to model weights and excludes bias terms, matching the benchmark protocol.

In [12]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    *,
    batch: int = 1024,
    shuffle: bool = False,
    seed: int = 1234,
) -> DataLoader:
    g = torch.Generator().manual_seed(int(seed))

    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).view(-1),
    )

    return DataLoader(
        ds,
        batch_size=int(batch),
        shuffle=bool(shuffle),
        generator=g,
        drop_last=False,
    )


def l2_penalty_weights_only(model: nn.Module) -> torch.Tensor:
    device = next(model.parameters()).device
    l2 = torch.zeros((), device=device)

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.endswith("bias") or name in ("bias", "b0", "intercept"):
            continue
        l2 = l2 + (p * p).sum()

    return l2


@torch.no_grad()
def bce_logits_loss(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
) -> float:
    model.eval()
    logits = model(X).view(-1)
    y = y.view(-1)
    loss = nn.BCEWithLogitsLoss(reduction="mean")(logits, y)
    return float(loss.detach().cpu().item())

# Step 5 — Fit BCE models and select the L2 penalty by inner cross-validation

This step fits logistic-regression models using binary cross-entropy and an explicit L2 penalty on the model weights. For each outer training fold, the L2 penalty is selected by inner stratified cross-validation on the training data only. The selected penalty is then used to refit the BCE model on the full outer training fold.

In [13]:
from sklearn.model_selection import StratifiedKFold


def fit_bce_lbfgs(
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    *,
    l2_lambda: float,
    max_iter: int = 500,
    tol_grad: float = 1e-7,
    tol_change: float = 1e-9,
    history_size: int = 100,
    device: str = "cpu",
):
    device_t = torch.device(device)

    X = torch.tensor(X_tr, dtype=torch.float32, device=device_t)
    y = torch.tensor(y_tr, dtype=torch.float32, device=device_t).view(-1)

    model = make_model_fn().to(device_t)
    bce = nn.BCEWithLogitsLoss(reduction="mean")

    opt = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=int(max_iter),
        tolerance_grad=float(tol_grad),
        tolerance_change=float(tol_change),
        history_size=int(history_size),
        line_search_fn="strong_wolfe",
    )

    l2_lambda_t = torch.tensor(float(l2_lambda), device=device_t)

    def closure():
        opt.zero_grad(set_to_none=True)

        logits = model(X).view(-1)
        loss = bce(logits, y) + l2_lambda_t * l2_penalty_weights_only(model)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite LBFGS loss: {loss.detach().item()}")

        loss.backward()
        return loss

    opt.step(closure)

    train_bce = bce_logits_loss(model, X, y)

    return model, train_bce


def select_l2_by_cv_bce(
    lambdas,
    *,
    make_model_fn,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    device: str = "cpu",
    n_splits: int = 5,
    lbfgs_max_iter: int = 500,
    seed: int = 1234,
):
    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    rows = []
    best_lambda = None
    best_cv_bce = float("inf")

    for lam in lambdas:
        fold_bces = []

        for fold_id, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y_tr)):
            X_tr_f = X_tr[tr_idx]
            y_tr_f = y_tr[tr_idx]
            X_va_f = X_tr[va_idx]
            y_va_f = y_tr[va_idx]

            model, _ = fit_bce_lbfgs(
                make_model_fn,
                X_tr_f,
                y_tr_f,
                l2_lambda=float(lam),
                max_iter=int(lbfgs_max_iter),
                device=device,
            )

            device_t = torch.device(device)
            X_va_t = torch.tensor(X_va_f, dtype=torch.float32, device=device_t)
            y_va_t = torch.tensor(y_va_f, dtype=torch.float32, device=device_t).view(-1)

            fold_bce = bce_logits_loss(model, X_va_t, y_va_t)
            fold_bces.append(fold_bce)

        mean_bce = float(np.mean(fold_bces))

        rows.append(
            {
                "l2_lambda": float(lam),
                "cv_bce_mean": mean_bce,
                "cv_bce_folds": fold_bces,
            }
        )

        if mean_bce < best_cv_bce:
            best_lambda = float(lam)
            best_cv_bce = mean_bce

    return best_lambda, best_cv_bce, rows

# Step 6 — Create the preprocessing variants

Create the five repeated train/test splits for the two preprocessing variants used in the Framingham experiments. The `mild` variant retains education as a single ordinal feature, whereas the `moderate` variant represents education using one-hot encoding.

In [14]:
splits_mild = make_framingham_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    device=DEVICE,
    fe_level="mild",
)

splits_moderate = make_framingham_splits_5x(
    model_df,
    target_col="FifteenYearCHD",
    base_seed=42,
    n_runs=5,
    device=DEVICE,
    fe_level="moderate",
)

print(f"Mild preprocessing     : {len(splits_mild)} splits")
print(f"Moderate preprocessing : {len(splits_moderate)} splits")
print(f"Device                 : {DEVICE}")

Mild preprocessing     : 5 splits
Moderate preprocessing : 5 splits
Device                 : cpu


# Step 7 — Define local BCE calibration helpers

This step defines local post-hoc calibration for the BCE logistic-regression model. For each threshold band, an adaptive calibration window is selected around the reference threshold using BCE training-fold predicted probabilities. The window is expanded until it contains at least 50 events and 50 non-events, or until the full probability range is used. Temperature scaling and Platt scaling are fitted only within this local BCE training window and applied only to BCE test observations whose original BCE predicted probabilities fall inside the same window.

In [15]:
def sigmoid_np(logits_np: np.ndarray) -> np.ndarray:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    return torch.sigmoid(logits_t).detach().cpu().numpy().astype(np.float64)


@torch.no_grad()
def predict_logits_torch(
    model: nn.Module,
    X_np: np.ndarray,
    *,
    device: str = DEVICE,
) -> np.ndarray:
    device_t = torch.device(device)
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32, device=device_t)
    return model(X_t).detach().cpu().view(-1).numpy().astype(np.float64)


def subset_counts(mask, y_np):
    y_sub = np.asarray(y_np).reshape(-1)[np.asarray(mask, dtype=bool)]
    n_pos = int(np.sum(y_sub == 1))
    n_neg = int(np.sum(y_sub == 0))
    return int(len(y_sub)), n_pos, n_neg


def find_local_calibration_range(
    probs_train: np.ndarray,
    y_train: np.ndarray,
    *,
    t_ref: float,
    start_half_width: float = LOCAL_START_HALF_WIDTH,
    expand_step: float = LOCAL_EXPAND_STEP,
    min_pos: int = LOCAL_MIN_POS,
    min_neg: int = LOCAL_MIN_NEG,
) -> dict:
    probs_train = np.asarray(probs_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)

    half_width = float(start_half_width)
    n_expand_steps = 0

    while True:
        low = max(0.0, float(t_ref) - half_width)
        high = min(1.0, float(t_ref) + half_width)

        mask = (probs_train >= low) & (probs_train <= high)
        n, n_pos, n_neg = subset_counts(mask, y_train)

        met_minimum = (n_pos >= int(min_pos)) and (n_neg >= int(min_neg))
        used_full_range = (low <= 0.0) and (high >= 1.0)

        if met_minimum or used_full_range:
            return {
                "low": float(low),
                "high": float(high),
                "half_width": float(half_width),
                "range_width": float(high - low),
                "n_expand_steps": int(n_expand_steps),
                "n": int(n),
                "n_pos": int(n_pos),
                "n_neg": int(n_neg),
                "met_minimum": bool(met_minimum),
                "used_full_range": bool(used_full_range),
                "mask": mask,
            }

        half_width += float(expand_step)
        n_expand_steps += 1


def fit_temperature_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = TEMP_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )
    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    log_temperature = torch.zeros(
        (),
        dtype=torch.float32,
        device=device_t,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        temperature = torch.exp(log_temperature).clamp(min=1e-6, max=1e6)
        loss = bce(logits / temperature, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite temperature loss: {loss.detach().item()}")

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).clamp(min=1e-6, max=1e6)
        nll_after = float(bce(logits / temperature, y).detach().cpu().item())

    return {
        "temperature": float(temperature.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_temperature_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (probs_reference_np >= float(low)) & (probs_reference_np <= float(high))
    logits_out[mask] = logits_out[mask] / float(temperature)

    return logits_out, mask


def fit_platt_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    max_iter: int = PLATT_MAX_ITER,
    device: str = DEVICE,
) -> dict:
    device_t = torch.device(device)

    logits = torch.tensor(
        np.asarray(logits_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )
    y = torch.tensor(
        np.asarray(y_np, dtype=np.float32).reshape(-1),
        device=device_t,
    )

    bce = nn.BCEWithLogitsLoss(reduction="mean")

    with torch.no_grad():
        nll_before = float(bce(logits, y).detach().cpu().item())

    slope = torch.ones((), dtype=torch.float32, device=device_t, requires_grad=True)
    intercept = torch.zeros((), dtype=torch.float32, device=device_t, requires_grad=True)

    optimizer = torch.optim.LBFGS(
        [slope, intercept],
        lr=0.1,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad(set_to_none=True)
        calibrated_logits = slope * logits + intercept
        loss = bce(calibrated_logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite Platt loss: {loss.detach().item()}")

        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        calibrated_logits = slope * logits + intercept
        nll_after = float(bce(calibrated_logits, y).detach().cpu().item())

    return {
        "platt_slope": float(slope.detach().cpu().item()),
        "platt_intercept": float(intercept.detach().cpu().item()),
        "nll_before": nll_before,
        "nll_after": nll_after,
    }


def apply_local_platt_to_logits(
    logits_np: np.ndarray,
    probs_reference_np: np.ndarray,
    *,
    low: float,
    high: float,
    slope: float,
    intercept: float,
) -> tuple[np.ndarray, np.ndarray]:
    logits_out = np.asarray(logits_np, dtype=np.float64).reshape(-1).copy()
    probs_reference_np = np.asarray(probs_reference_np, dtype=float).reshape(-1)

    mask = (probs_reference_np >= float(low)) & (probs_reference_np <= float(high))
    logits_out[mask] = float(slope) * logits_out[mask] + float(intercept)

    return logits_out, mask


def evaluate_nb_from_logits_np(
    logits_np: np.ndarray,
    y_np: np.ndarray,
    *,
    thresh_min: float,
    thresh_max: float,
    num_points: int = TEST_RANGE_POINTS,
) -> float:
    logits_t = torch.tensor(np.asarray(logits_np, dtype=np.float32).reshape(-1))
    y_t = torch.tensor(np.asarray(y_np, dtype=np.float32).reshape(-1))

    return float(
        average_nb_over_range(
            logits_t,
            y_t,
            thresh_min=float(thresh_min),
            thresh_max=float(thresh_max),
            num_points=int(num_points),
            input_is_logit=True,
            method="mean",
        )
    )

# Step 8 — Run the complete Framingham logistic-regression experiment

This step runs the full logistic-regression experiment across both preprocessing variants, all five outer train/test splits, and the three threshold bands. For each split and band, we fit a BCE logistic-regression model, fine-tune a copy using smooth Net Benefit, and then apply local post-hoc temperature and Platt calibration. Local calibration is fitted on training predictions within an adaptive probability window around the decision threshold and applied only to test observations whose original predicted risk falls inside the same window.

In [16]:
from copy import deepcopy
from nbloss.trainer import nb_anneal_only_with_l2

L2_LAM_GRID = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]

DATASETS = {
    "mild": splits_mild,
    "moderate": splits_moderate,
}

results = []


def add_result_row(
    *,
    dataset_name: str,
    run_id: int,
    band_name: str,
    model_name: str,
    split_pack,
    t_ref: float,
    t_min: float,
    t_max: float,
    l2_lambda: float,
    cv_bce: float,
    train_bce: float,
    test_nb: float,
    delta_vs_bce: float,
    local_info: dict | None = None,
    calibration_info: dict | None = None,
):
    local_info = local_info or {}
    calibration_info = calibration_info or {}

    results.append(
        {
            "dataset": dataset_name,
            "run_id": int(run_id),
            "band_name": band_name,
            "model": model_name,
            "prev_train": float(split_pack.y_train_mean),
            "prev_test": float(split_pack.y_test_mean),
            "t_ref": float(t_ref),
            "t_min": float(t_min),
            "t_max": float(t_max),
            "l2_lambda": float(l2_lambda),
            "cv_bce": float(cv_bce),
            "train_bce": float(train_bce),
            "test_nb": float(test_nb),
            "delta_vs_bce": float(delta_vs_bce),
            "local_low": local_info.get("low", np.nan),
            "local_high": local_info.get("high", np.nan),
            "local_half_width": local_info.get("half_width", np.nan),
            "local_range_width": local_info.get("range_width", np.nan),
            "local_expand_steps": local_info.get("n_expand_steps", np.nan),
            "local_train_n": local_info.get("n", np.nan),
            "local_train_pos": local_info.get("n_pos", np.nan),
            "local_train_neg": local_info.get("n_neg", np.nan),
            "local_met_minimum": local_info.get("met_minimum", np.nan),
            "local_used_full_range": local_info.get("used_full_range", np.nan),
            "temperature": calibration_info.get("temperature", np.nan),
            "platt_slope": calibration_info.get("platt_slope", np.nan),
            "platt_intercept": calibration_info.get("platt_intercept", np.nan),
            "train_local_nll_before": calibration_info.get("nll_before", np.nan),
            "train_local_nll_after": calibration_info.get("nll_after", np.nan),
            "n_features": len(split_pack.feature_cols),
        }
    )


for dataset_name, splits_list in DATASETS.items():
    print(f"\n==================== DATASET: {dataset_name} ====================")

    for sp in splits_list:
        run_id = int(sp.run_id)
        split_seed = int(sp.seed)

        Xtr = np.asarray(sp.X_train, dtype=np.float32)
        Xte = np.asarray(sp.X_test, dtype=np.float32)

        ytr = np.asarray(sp.y_train, dtype=np.float32).reshape(-1)
        yte = np.asarray(sp.y_test, dtype=np.float32).reshape(-1)

        prev_train = float(ytr.mean())

        band_specs = []

        for t_ref in (0.05, 0.10, 0.20):
            t_min, t_max = band_from_t_ref(t_ref)
            band_specs.append((f"t_{t_ref:.2f}", t_ref, t_min, t_max))

        d_in = int(Xtr.shape[1])

        def make_lr_model():
            return TorchLR(d_in)

        lam_star, best_cv_bce, cv_grid = select_l2_by_cv_bce(
            L2_LAM_GRID,
            make_model_fn=make_lr_model,
            X_tr=Xtr,
            y_tr=ytr,
            device=DEVICE,
            n_splits=5,
            lbfgs_max_iter=500,
            seed=split_seed,
        )

        print(
            f"\n[{dataset_name} | run {run_id}] "
            f"prev_train={prev_train:.4f} | "
            f"selected l2_lambda={lam_star:g} | "
            f"best inner-CV BCE={best_cv_bce:.6f}"
        )

        train_dl = make_loader(
            Xtr,
            ytr,
            batch=1024,
            shuffle=True,
            seed=split_seed,
        )

        for band_name, t_ref, t_min, t_max in band_specs:
            print(
                f"\n[{dataset_name} | run {run_id}] "
                f"=== BAND: {band_name} "
                f"(t_ref={t_ref:.4f}, [{t_min:.4f}, {t_max:.4f}]) ==="
            )

            bce_model, train_bce = fit_bce_lbfgs(
                make_lr_model,
                Xtr,
                ytr,
                l2_lambda=float(lam_star),
                max_iter=500,
                device=DEVICE,
            )

            logits_bce_train = predict_logits_torch(bce_model, Xtr, device=DEVICE)
            logits_bce_test = predict_logits_torch(bce_model, Xte, device=DEVICE)

            probs_bce_train = sigmoid_np(logits_bce_train)
            probs_bce_test = sigmoid_np(logits_bce_test)

            nb_bce = evaluate_nb_from_logits_np(
                logits_bce_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{band_name}] BCE-LR NB={nb_bce:.6f}")

            local_bce = find_local_calibration_range(
                probs_bce_train,
                ytr,
                t_ref=float(t_ref),
                start_half_width=LOCAL_START_HALF_WIDTH,
                expand_step=LOCAL_EXPAND_STEP,
                min_pos=LOCAL_MIN_POS,
                min_neg=LOCAL_MIN_NEG,
            )

            logits_bce_train_local = logits_bce_train[local_bce["mask"]]
            ytr_bce_local = ytr[local_bce["mask"]]

            temp_bce_fit = fit_temperature_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=TEMP_MAX_ITER,
                device=DEVICE,
            )

            platt_bce_fit = fit_platt_from_logits_np(
                logits_bce_train_local,
                ytr_bce_local,
                max_iter=PLATT_MAX_ITER,
                device=DEVICE,
            )

            logits_bce_temp_test, _ = apply_local_temperature_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                temperature=temp_bce_fit["temperature"],
            )

            logits_bce_platt_test, _ = apply_local_platt_to_logits(
                logits_bce_test,
                probs_bce_test,
                low=local_bce["low"],
                high=local_bce["high"],
                slope=platt_bce_fit["platt_slope"],
                intercept=platt_bce_fit["platt_intercept"],
            )

            nb_bce_temp = evaluate_nb_from_logits_np(
                logits_bce_temp_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            nb_bce_platt = evaluate_nb_from_logits_np(
                logits_bce_platt_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            snb_start = TorchLR(d_in).to(DEVICE)
            snb_start.load_state_dict(
                {
                    k: v.detach().cpu().clone()
                    for k, v in bce_model.state_dict().items()
                }
            )

            snb_model = nb_anneal_only_with_l2(
                snb_start,
                train_dl,
                thresh_min=float(t_min),
                thresh_max=float(t_max),
                num_points_train=int(TRAIN_RANGE_POINTS),
                inverse_temps=tuple(INVERSE_TEMPS),
                epochs_per_temp=int(EPOCHS_PER_TEMP),
                patience_hard=int(PATIENCE_HARD),
                hard_range_num_points=int(TEST_RANGE_POINTS),
                lr_adam=float(LR_ADAMW),
                l2_lambda=float(lam_star),
                penalty_fn=l2_penalty_weights_only,
                device=DEVICE,
                seed=int(MODEL_SEED) + 1000 * run_id + 17,
                log_every=20,
            )

            logits_snb_test = predict_logits_torch(snb_model, Xte, device=DEVICE)

            nb_snb = evaluate_nb_from_logits_np(
                logits_snb_test,
                yte,
                thresh_min=t_min,
                thresh_max=t_max,
                num_points=TEST_RANGE_POINTS,
            )

            print(f"[TEST][{band_name}] SNB-LR NB={nb_snb:.6f} | Δ={nb_snb - nb_bce:+.6f}")

            add_result_row(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_lr",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce,
                delta_vs_bce=0.0,
            )

            add_result_row(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_lr_local_temperature",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_temp,
                delta_vs_bce=nb_bce_temp - nb_bce,
                local_info=local_bce,
                calibration_info=temp_bce_fit,
            )

            add_result_row(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="bce_lr_local_platt",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_bce_platt,
                delta_vs_bce=nb_bce_platt - nb_bce,
                local_info=local_bce,
                calibration_info=platt_bce_fit,
            )

            add_result_row(
                dataset_name=dataset_name,
                run_id=run_id,
                band_name=band_name,
                model_name="snb_lr",
                split_pack=sp,
                t_ref=t_ref,
                t_min=t_min,
                t_max=t_max,
                l2_lambda=lam_star,
                cv_bce=best_cv_bce,
                train_bce=train_bce,
                test_nb=nb_snb,
                delta_vs_bce=nb_snb - nb_bce,
            )

            print(
                f"[CAL][{band_name}] "
                f"BCE-temp Δ={nb_bce_temp - nb_bce:+.6f} | "
                f"BCE-Platt Δ={nb_bce_platt - nb_bce:+.6f} | "
                f"SNB Δ={nb_snb - nb_bce:+.6f}"
            )

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)

print("\nFinished.")
print("results_df shape:", results_df.shape)
display(results_df.head())


==================== DATASET: mild ====================

[mild | run 0] prev_train=0.1667 | selected l2_lambda=0.001 | best inner-CV BCE=0.388405

[mild | run 0] === BAND: t_0.05 (t_ref=0.0500, [0.0250, 0.0750]) ===
[TEST][t_0.05] BCE-LR NB=0.127033
[SNB start] initial train hard NB range = 0.127590
[SNB inverse_temp=1] epoch 020 | train_loss=-0.112331 | train_hard_nb=0.126602
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=1 no improvement greater than epsilon_nb=1e-12; keep global train hard NB: 0.127590
[SNB inverse_temp=4] epoch 020 | train_loss=-0.124991 | train_hard_nb=0.127709
[SNB inverse_temp=4] epoch 040 | train_loss=-0.125157 | train_hard_nb=0.127687
>>> Early stop inverse-temperature phase.
[commit] inverse_temp=4 improved global train hard NB: 0.127590 → 0.127716
[SNB inverse_temp=10] epoch 020 | train_loss=-0.126566 | train_hard_nb=0.128225
[SNB inverse_temp=10] epoch 040 | train_loss=-0.126803 | train_hard_nb=0.128392
[SNB inverse_temp=10] epoch 060 | tr

,dataset,run_id,band_name,model,prev_train,prev_test,t_ref,t_min,t_max,l2_lambda,...,local_train_pos,local_train_neg,local_met_minimum,local_used_full_range,temperature,platt_slope,platt_intercept,train_local_nll_before,train_local_nll_after,n_features
0,mild,0,t_0.05,bce_lr,0.166667,0.166446,0.05,0.025,0.075,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
1,mild,0,t_0.05,bce_lr_local_temperature,0.166667,0.166446,0.05,0.025,0.075,0.001,...,115.0,1615.0,True,False,0.952778,NaN,NaN,0.235154,0.234686,15
2,mild,0,t_0.05,bce_lr_local_platt,0.166667,0.166446,0.05,0.025,0.075,0.001,...,115.0,1615.0,True,False,NaN,0.989995,-0.150996,0.235154,0.234652,15
3,mild,0,t_0.05,snb_lr,0.166667,0.166446,0.05,0.025,0.075,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
4,mild,0,t_0.10,bce_lr,0.166667,0.166446,0.10,0.075,0.125,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15


In [17]:
from scipy.stats import ttest_rel


print("\n======================================")
print("Finished. results_df shape:")
print(results_df.shape)
print("======================================")
display(results_df.head())


paired_df = (
    results_df
    .pivot_table(
        index=["dataset", "run_id", "band_name"],
        columns="model",
        values="test_nb",
        aggfunc="first",
    )
    .reset_index()
)

baseline_model = "bce_lr"

summary_rows = []

for (dataset_name, band_name), sub in paired_df.groupby(["dataset", "band_name"]):
    for model_name in [
        "bce_lr_local_temperature",
        "bce_lr_local_platt",
        "snb_lr",
    ]:
        if model_name not in sub.columns:
            continue

        tmp = sub.dropna(subset=[baseline_model, model_name]).copy()
        n_runs = len(tmp)

        if n_runs == 0:
            continue

        delta = tmp[model_name] - tmp[baseline_model]

        if n_runs >= 2:
            t_stat, p_value = ttest_rel(tmp[model_name], tmp[baseline_model])
            t_stat = float(t_stat)
            p_value = float(p_value)
        else:
            t_stat = np.nan
            p_value = np.nan

        summary_rows.append(
            {
                "dataset": dataset_name,
                "band_name": band_name,
                "model": model_name,
                "n_runs": int(n_runs),
                "mean_bce_lr": float(tmp[baseline_model].mean()),
                "mean_model_nb": float(tmp[model_name].mean()),
                "mean_delta_vs_bce": float(delta.mean()),
                "sd_delta_vs_bce": float(delta.std(ddof=1)) if n_runs > 1 else np.nan,
                "wins_vs_bce": int((delta > 0).sum()),
                "losses_vs_bce": int((delta < 0).sum()),
                "ties_vs_bce": int((delta == 0).sum()),
                "paired_t_vs_bce": t_stat,
                "paired_p_vs_bce": p_value,
                "significant_0.05": bool(p_value < 0.05) if np.isfinite(p_value) else False,
            }
        )

summary_tests_df = pd.DataFrame(summary_rows)

print("\n======================================")
print("Paired summary versus BCE-LR across 5 runs")
print("======================================")
display(summary_tests_df)


summary_df = (
    results_df
    .groupby(["dataset", "band_name", "model"], as_index=False)
    .agg(
        mean_test_nb=("test_nb", "mean"),
        sd_test_nb=("test_nb", "std"),
        mean_delta_vs_bce=("delta_vs_bce", "mean"),
        sd_delta_vs_bce=("delta_vs_bce", "std"),
        mean_local_low=("local_low", "mean"),
        mean_local_high=("local_high", "mean"),
        mean_local_range_width=("local_range_width", "mean"),
        mean_local_train_n=("local_train_n", "mean"),
        mean_local_train_pos=("local_train_pos", "mean"),
        mean_local_train_neg=("local_train_neg", "mean"),
        mean_temperature=("temperature", "mean"),
        mean_platt_slope=("platt_slope", "mean"),
        mean_platt_intercept=("platt_intercept", "mean"),
        mean_train_local_nll_before=("train_local_nll_before", "mean"),
        mean_train_local_nll_after=("train_local_nll_after", "mean"),
        n_runs=("run_id", "nunique"),
    )
)

print("\n======================================")
print("Mean results across 5 runs")
print("======================================")
display(summary_df)


Finished. results_df shape:
(120, 30)


,dataset,run_id,band_name,model,prev_train,prev_test,t_ref,t_min,t_max,l2_lambda,...,local_train_pos,local_train_neg,local_met_minimum,local_used_full_range,temperature,platt_slope,platt_intercept,train_local_nll_before,train_local_nll_after,n_features
0,mild,0,t_0.05,bce_lr,0.166667,0.166446,0.05,0.025,0.075,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
1,mild,0,t_0.05,bce_lr_local_temperature,0.166667,0.166446,0.05,0.025,0.075,0.001,...,115.0,1615.0,True,False,0.952778,NaN,NaN,0.235154,0.234686,15
2,mild,0,t_0.05,bce_lr_local_platt,0.166667,0.166446,0.05,0.025,0.075,0.001,...,115.0,1615.0,True,False,NaN,0.989995,-0.150996,0.235154,0.234652,15
3,mild,0,t_0.05,snb_lr,0.166667,0.166446,0.05,0.025,0.075,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
4,mild,0,t_0.10,bce_lr,0.166667,0.166446,0.10,0.075,0.125,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15



Paired summary versus BCE-LR across 5 runs


,dataset,band_name,model,n_runs,mean_bce_lr,mean_model_nb,mean_delta_vs_bce,sd_delta_vs_bce,wins_vs_bce,losses_vs_bce,ties_vs_bce,paired_t_vs_bce,paired_p_vs_bce,significant_0.05
0,mild,t_0.05,bce_lr_local_temperature,5,0.127252,0.127517,0.000265,0.000164,5,0,0,3.595917,0.022841,True
1,mild,t_0.05,bce_lr_local_platt,5,0.127252,0.127591,0.000339,0.000242,5,0,0,3.140827,0.034820,True
2,mild,t_0.05,snb_lr,5,0.127252,0.127175,-0.000077,0.000764,2,3,0,-0.226505,0.831913,False
3,mild,t_0.10,bce_lr_local_temperature,5,0.096991,0.096879,-0.000112,0.000493,2,3,0,-0.506749,0.638989,False
4,mild,t_0.10,bce_lr_local_platt,5,0.096991,0.097007,0.000016,0.000400,2,3,0,0.090323,0.932373,False
5,mild,t_0.10,snb_lr,5,0.096991,0.095802,-0.001189,0.001867,2,3,0,-1.423799,0.227606,False
6,mild,t_0.20,bce_lr_local_temperature,5,0.051631,0.051494,-0.000138,0.000427,2,3,0,-0.720261,0.511207,False
7,mild,t_0.20,bce_lr_local_platt,5,0.051631,0.051434,-0.000197,0.000601,3,2,0,-0.734411,0.503428,False
8,mild,t_0.20,snb_lr,5,0.051631,0.047463,-0.004169,0.005570,2,3,0,-1.673564,0.169530,False
9,moderate,t_0.05,bce_lr_local_temperature,5,0.127071,0.127357,0.000286,0.000207,5,0,0,3.088258,0.036636,True



Mean results across 5 runs


,dataset,band_name,model,mean_test_nb,sd_test_nb,mean_delta_vs_bce,sd_delta_vs_bce,mean_local_low,mean_local_high,mean_local_range_width,mean_local_train_n,mean_local_train_pos,mean_local_train_neg,mean_temperature,mean_platt_slope,mean_platt_intercept,mean_train_local_nll_before,mean_train_local_nll_after,n_runs
0,mild,t_0.05,bce_lr,0.127252,0.001225,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
1,mild,t_0.05,bce_lr_local_platt,0.127591,0.001443,0.000339,0.000242,0.0,0.15,0.15,1719.6,122.2,1597.4,NaN,1.118096,0.206256,0.244198,0.243813,5
2,mild,t_0.05,bce_lr_local_temperature,0.127517,0.001350,0.000265,0.000164,0.0,0.15,0.15,1719.6,122.2,1597.4,0.968509,NaN,NaN,0.244198,0.243953,5
3,mild,t_0.05,snb_lr,0.127175,0.001840,-0.000077,0.000764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
4,mild,t_0.10,bce_lr,0.096991,0.003935,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
5,mild,t_0.10,bce_lr_local_platt,0.097007,0.004129,0.000016,0.000400,0.0,0.20,0.20,2099.2,189.6,1909.6,NaN,1.125867,0.225353,0.284383,0.284037,5
6,mild,t_0.10,bce_lr_local_temperature,0.096879,0.004012,-0.000112,0.000493,0.0,0.20,0.20,2099.2,189.6,1909.6,0.976807,NaN,NaN,0.284383,0.284260,5
7,mild,t_0.10,snb_lr,0.095802,0.003962,-0.001189,0.001867,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
8,mild,t_0.20,bce_lr,0.051631,0.005632,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
9,mild,t_0.20,bce_lr_local_platt,0.051434,0.005181,-0.000197,0.000601,0.1,0.30,0.20,1367.6,250.4,1117.2,NaN,0.987850,0.006014,0.465944,0.465845,5
